In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ---------- pretrained_inference.py ----------
# !pip install ultralytics opencv-python pandas

from ultralytics import YOLO
import cv2
import os
import pandas as pd

# Load pretrained YOLOv8 model (COCO pretrained)
model = YOLO("yolov8s.pt")

# Paths
input_folder = "/content/drive/MyDrive/Tomato Yield Estimation RE/Tomato-Images/Tomato"
output_folder = "/content/drive/MyDrive/Tomato Yield Estimation RE/Bounded_Joshi"
os.makedirs(output_folder, exist_ok=True)

# CSV to save bounding box data
csv_data = []

for img_name in os.listdir(input_folder):
    if not img_name.lower().endswith(('.jpg', '.png', '.jpeg')):
        continue

    img_path = os.path.join(input_folder, img_name)
    results = model(img_path)
    img = cv2.imread(img_path)

    for r in results:
        boxes = r.boxes.xyxy.cpu().numpy()  # [x1, y1, x2, y2]
        confs = r.boxes.conf.cpu().numpy()
        clss = r.boxes.cls.cpu().numpy()

        for box, conf, cls_id in zip(boxes, confs, clss):
            x1, y1, x2, y2 = map(int, box)
            class_name = model.names[int(cls_id)]
            label = f"{class_name} {conf:.2f}"

            # Draw bounding box
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(img, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

            # Save box data
            csv_data.append([img_name, x1, y1, x2, y2, class_name, conf])

    # Save image with boxes
    save_path = os.path.join(output_folder, img_name)
    cv2.imwrite(save_path, img)
    print(f"Saved: {save_path}")

# Save detections to CSV
df = pd.DataFrame(csv_data, columns=["image", "x1", "y1", "x2", "y2", "class", "confidence"])
df.to_csv(os.path.join(output_folder, "detections.csv"), index=False)
print("✅ All results saved in 'results/' folder and detections.csv")



image 1/1 /content/drive/MyDrive/Tomato Yield Estimation RE/Tomato-Images/Tomato/IMG_1838.JPG: 640x480 (no detections), 57.0ms
Speed: 12.5ms preprocess, 57.0ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 480)
Saved: /content/drive/MyDrive/Tomato Yield Estimation RE/Bounded_Joshi/IMG_1838.JPG

image 1/1 /content/drive/MyDrive/Tomato Yield Estimation RE/Tomato-Images/Tomato/IMG_1843.JPG: 640x480 1 tie, 1 book, 12.3ms
Speed: 4.5ms preprocess, 12.3ms inference, 2.3ms postprocess per image at shape (1, 3, 640, 480)
Saved: /content/drive/MyDrive/Tomato Yield Estimation RE/Bounded_Joshi/IMG_1843.JPG

image 1/1 /content/drive/MyDrive/Tomato Yield Estimation RE/Tomato-Images/Tomato/IMG_1816.JPG: 640x480 2 books, 12.2ms
Speed: 2.8ms preprocess, 12.2ms inference, 1.6ms postprocess per image at shape (1, 3, 640, 480)
Saved: /content/drive/MyDrive/Tomato Yield Estimation RE/Bounded_Joshi/IMG_1816.JPG

image 1/1 /content/drive/MyDrive/Tomato Yield Estimation RE/Tomato-Images/Tomato/

In [ ]:
import os, shutil, random, csv
from pathlib import Path
import pandas as pd
from PIL import Image

# ---- PATHS (update these) ----
IMAGES_DIR = "/content/drive/MyDrive/Tomato Yield Estimation RE/Tomato-Images/Tomato"          # folder with all tomato images
CSV_PATH   = "/content/drive/MyDrive/Tomato Yield Estimation RE/tomato.csv" # csv with bbox or weights
OUTPUT_DIR = "/content/drive/MyDrive/Tomato Yield Estimation RE/Bounded_Joshi/attempt_2"
VAL_SPLIT  = 0.2

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- READ CSV ----
df = pd.read_csv(CSV_PATH)
print("CSV Columns:", df.columns.tolist())

# Normalize column names
df.columns = [c.strip().lower() for c in df.columns]
image_col = None
for col in df.columns:
    if "image" in col:
        image_col = col
        break

if not image_col:
    raise SystemExit("❌ Could not detect an 'Image Name' column. Check your CSV headers!")

weight_col = "weight" if "weight" in df.columns else None
tomato_id_col = "tomato id" if "tomato id" in df.columns else None
view_col = "view" if "view" in df.columns else None

print(f"Detected image column: {image_col}")
print(f"Detected weight column: {weight_col}")
print(f"Detected tomato ID column: {tomato_id_col}")
print(f"Detected view column: {view_col}")

# ---- COLLECT IMAGES ----
all_imgs = [f for f in os.listdir(IMAGES_DIR) if f.lower().endswith(('.jpg','.jpeg','.png'))]
print("Found", len(all_imgs), "image files in directory.")

# ---- Train/Val split ----
random.seed(42)
random.shuffle(all_imgs)
n_val = max(1, int(len(all_imgs) * VAL_SPLIT))
val_set = set(all_imgs[:n_val])
train_set = set(all_imgs[n_val:])

# ---- Make output dirs ----
def make_dirs(base):
    for split in ("train","val"):
        (Path(base)/split/"images").mkdir(parents=True, exist_ok=True)
        (Path(base)/split/"labels").mkdir(parents=True, exist_ok=True)

make_dirs(OUTPUT_DIR)

# ---- Copy images & make empty label files ----
for img_name in all_imgs:
    split = "val" if img_name in val_set else "train"
    src = os.path.join(IMAGES_DIR, img_name)
    dst_img = os.path.join(OUTPUT_DIR, split, "images", img_name)
    shutil.copy(src, dst_img)
    label_name = os.path.splitext(img_name)[0] + ".txt"
    dst_label = os.path.join(OUTPUT_DIR, split, "labels", label_name)
    open(dst_label, "w").close()  # empty YOLO label (no boxes yet)

# ---- Save image-weight mapping ----
if weight_col:
    weights_path = os.path.join(OUTPUT_DIR, "image_weights.csv")
    with open(weights_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["image", "weight", "tomato_id", "view"])
        for _, row in df.iterrows():
            img = str(row[image_col])
            wt = row[weight_col]
            tid = row[tomato_id_col] if tomato_id_col else ""
            view = row[view_col] if view_col else ""
            writer.writerow([img, wt, tid, view])
    print("✅ Saved image_weights.csv at:", weights_path)

# ---- Create YOLO data.yaml ----
yaml_path = "/content/data.yaml"
with open(yaml_path, "w") as f:
    f.write(f"train: {OUTPUT_DIR}/train/images\nval: {OUTPUT_DIR}/val/images\nnc: 1\nnames: ['tomato']\n")

print("✅ YOLO dataset folder created at:", OUTPUT_DIR)
print("✅ data.yaml saved at:", yaml_path)

# Preview dataset split
print("Train images:", len(os.listdir(f'{OUTPUT_DIR}/train/images')))
print("Val images:", len(os.listdir(f'{OUTPUT_DIR}/val/images')))

CSV Columns: ['Image Name', 'Weight', 'View', 'Tomato ID']
Detected image column: image name
Detected weight column: weight
Detected tomato ID column: tomato id
Detected view column: view
Found 45 image files in directory.
✅ Saved image_weights.csv at: /content/drive/MyDrive/Tomato Yield Estimation RE/Bounded_Joshi/attempt_2/image_weights.csv
✅ YOLO dataset folder created at: /content/drive/MyDrive/Tomato Yield Estimation RE/Bounded_Joshi/attempt_2
✅ data.yaml saved at: /content/data.yaml
Train images: 36
Val images: 9


In [2]:
# ===============================================================
# 🍅 1. Convert Supervisely JSON annotations to YOLO format
# ===============================================================
!pip install ultralytics Pillow tqdm --quiet

import json, os, shutil, random
from pathlib import Path
from PIL import Image
from tqdm import tqdm

# ---- CONFIG ----
IMAGES_DIR = "/content/drive/MyDrive/Tomato Yield Estimation RE/tomato-detection-DatasetNinja/ds0/img"          # folder with tomato images
ANN_DIR    = "/content/drive/MyDrive/Tomato Yield Estimation RE/tomato-detection-DatasetNinja/ds0/ann"     # folder with Supervisely JSONs
OUT_DIR    = "/content/drive/MyDrive/Tomato Yield Estimation RE/Bounded_Joshi/attempt_3"         # output YOLO dataset
VAL_SPLIT  = 0.2                        # fraction of validation set
CLASS_NAME = "tomato"

# ---- CREATE FOLDERS ----
for split in ("train","val"):
    (Path(OUT_DIR)/split/"images").mkdir(parents=True, exist_ok=True)
    (Path(OUT_DIR)/split/"labels").mkdir(parents=True, exist_ok=True)

# ---- HELPER FUNCTION ----
def convert_box(w, h, x1, y1, x2, y2):
    """Convert pixel coords to normalized YOLO (x_center, y_center, w, h)."""
    x1, y1, x2, y2 = float(x1), float(y1), float(x2), float(y2)
    bw, bh = x2 - x1, y2 - y1
    xc, yc = x1 + bw / 2, y1 + bh / 2
    return xc / w, yc / h, bw / w, bh / h

# ---- GATHER FILES ----
json_files = [f for f in os.listdir(ANN_DIR) if f.endswith(".json")]
random.seed(42)
random.shuffle(json_files)
n_val = max(1, int(len(json_files) * VAL_SPLIT))
val_set = set(json_files[:n_val])

print(f"Found {len(json_files)} annotation files. Splitting {len(json_files)-len(val_set)} train / {len(val_set)} val.")

# ---- CONVERT ----
for jf in tqdm(json_files):
    jp = os.path.join(ANN_DIR, jf)
    with open(jp, "r") as f:
        data = json.load(f)

    img_name = jf.replace(".json", "")
    img_path = os.path.join(IMAGES_DIR, img_name)
    if not os.path.exists(img_path):
        # Try fallback if extension mismatch (.png, .jpg)
        for ext in [".png", ".jpg", ".jpeg"]:
            if os.path.exists(img_path + ext):
                img_path = img_path + ext
                break
        else:
            print("⚠️ Image not found for", jf)
            continue

    with Image.open(img_path) as im:
        w, h = im.size

    yolo_lines = []
    for obj in data.get("objects", []):
        if obj.get("geometryType") != "rectangle":
            continue
        (x1, y1), (x2, y2) = obj["points"]["exterior"]
        xc, yc, bw, bh = convert_box(w, h, x1, y1, x2, y2)
        yolo_lines.append(f"0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")

    # Skip empty labels if no tomatoes (shouldn't happen)
    label_name = os.path.splitext(os.path.basename(img_path))[0] + ".txt"
    split = "val" if jf in val_set else "train"

    shutil.copy(img_path, os.path.join(OUT_DIR, split, "images", os.path.basename(img_path)))
    with open(os.path.join(OUT_DIR, split, "labels", label_name), "w") as f:
        f.write("\n".join(yolo_lines))

print("✅ Conversion complete!")

# ---- WRITE data.yaml ----
yaml_path = "/content/data.yaml"
with open(yaml_path, "w") as f:
    f.write(f"train: {OUT_DIR}/train/images\nval: {OUT_DIR}/val/images\nnc: 1\nnames: ['{CLASS_NAME}']\n")
print("✅ data.yaml saved at", yaml_path)

# ===============================================================
# 🧠 2. Train YOLOv8 on the converted dataset
# ===============================================================
from ultralytics import YOLO

model = YOLO("yolov8s.pt")  # pretrained COCO weights
results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    lr0=0.001,
    project="/content/tomato_training",
    name="exp1",
    patience=10
)

print(f"✅ Training finished! Best weights at:{results.save_dir}/weights/best.pt")

# ===============================================================
# 🎯 3. Test on validation images
# ===============================================================
trained_model = YOLO(f"{results.save_dir}/weights/best.pt")
pred = trained_model.predict(source=f"{OUT_DIR}/val/images", conf=0.3, save=True)
print("✅ Predictions saved under:", pred[0].save_dir)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 30.8 MB/s eta 0:00:00
Found 907 annotation files. Splitting 726 train / 181 val.


  3%|▎         | 29/907 [00:25<02:49,  5.17it/s]

⚠️ Image not found for tomato296 (1).png.json


  6%|▌         | 54/907 [00:32<03:40,  3.87it/s]

⚠️ Image not found for tomato291 (1).png.json


 12%|█▏        | 106/907 [00:46<04:01,  3.32it/s]

... (18 outputs left out)

⚠️ Image not found for tomato293 (1).png.json


100%|██████████| 907/907 [04:39<00:00,  3.25it/s]


✅ Conversion complete!
✅ data.yaml saved at /content/data.yaml
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.3.228 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.

TypeError: unsupported operand type(s) for +: 'PosixPath' and 'str'

In [4]:
print(f"✅ Training finished! Best weights at:{results.save_dir}/weights/best.pt")

# ===============================================================
# 🎯 3. Test on validation images
# ===============================================================
trained_model = YOLO(f"{results.save_dir}/weights/best.pt")
pred = trained_model.predict(source=f"{OUT_DIR}/val/images", conf=0.3, save=True)
print("✅ Predictions saved under:", pred[0].save_dir)


✅ Training finished! Best weights at:/content/tomato_training/exp1/weights/best.pt

image 1/177 /content/drive/MyDrive/Tomato Yield Estimation RE/Bounded_Joshi/attempt_3/val/images/tomato100.png: 640x512 1 tomato, 84.8ms
image 2/177 /content/drive/MyDrive/Tomato Yield Estimation RE/Bounded_Joshi/attempt_3/val/images/tomato115.png: 640x512 1 tomato, 27.5ms
image 3/177 /content/drive/MyDrive/Tomato Yield Estimation RE/Bounded_Joshi/attempt_3/val/images/tomato118.png: 512x640 11 tomatos, 46.9ms
image 4/177 /content/drive/MyDrive/Tomato Yield Estimation RE/Bounded_Joshi/attempt_3/val/images/tomato123.png: 512x640 7 tomatos, 13.0ms
image 5/177 /content/drive/MyDrive/Tomato Yield Estimation RE/Bounded_Joshi/attempt_3/val/images/tomato124.png: 640x512 12 tomatos, 13.0ms
image 6/177 /content/drive/MyDrive/Tomato Yield Estimation RE/Bounded_Joshi/attempt_3/val/images/tomato127.png: 512x640 2 tomatos, 13.8ms
image 7/177 /content/drive/MyDrive/Tomato Yield Estimation RE/Bounded_Joshi/attempt_3/va